In [ ]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
import pandas as pd

split_path = '/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/raw_3'

train_pl = pd.read_csv(f'{split_path}/train.csv')
test_1_pl = pd.read_csv(f'{split_path}/test_p1.csv')
test_2_pl = pd.read_csv(f'{split_path}/test_p2.csv')
test_3_pl = pd.read_csv(f'{split_path}/test_p3.csv')
test_4_pl = pd.read_csv(f'{split_path}/test_p4.csv')
val = pd.read_csv(f'{split_path}/val_p4.csv')

In [ ]:
import pandas as pd
import polars as pl

print('Checking for non-numeric columns in train_pl:')
string_columns = []
for col in train_pl.columns:
    if not pd.api.types.is_numeric_dtype(train_pl[col].dtype):
        print(f"Column '{col}' has type: {train_pl[col].dtype}")
        string_columns.append(col)

if not string_columns:
    print("No non-numeric columns found that need type conversion.")
else:
    print(f"Found non-numeric columns: {string_columns}. Please review if these need to be converted to a numeric type for extra-trees imputation.")

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

exclude_cols = ['user_id', 'course_id', 'label_3']
numeric_cols = [col for col in train_pl.columns
                if pd.api.types.is_numeric_dtype(train_pl[col].dtype) and col not in exclude_cols]

imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=1, max_depth=5),
    max_iter=1,
    sample_posterior=False,
    random_state=42
)

print(f"Calculated numeric columns for imputation: {len(numeric_cols)} columns.")

In [ ]:
dataframes_for_imputation = [
    (train_pl, 'train_pl'),
    (test_1_pl, 'test_1_pl'),
    (test_2_pl, 'test_2_pl'),
    (test_3_pl, 'test_3_pl'),
    (test_4_pl, 'test_4_pl'),
    (val, 'val')
]

for df_obj, df_name in dataframes_for_imputation:
    print(f"Imputing missing values in {df_name}...")

    columns_to_impute = [
        col for col in df_obj.columns
        if pd.api.types.is_numeric_dtype(df_obj[col].dtype) and df_obj[col].isnull().any() and col in numeric_cols
    ]

    if columns_to_impute:
        if df_name == 'train_pl':
            df_obj[numeric_cols] = imputer.fit_transform(df_obj[numeric_cols])
        else:
            df_obj[numeric_cols] = imputer.transform(df_obj[numeric_cols])
    else:
        print(f"No numeric columns with missing values found in {df_name} for imputation.")

    if df_name == 'train_pl':
        train_pl = df_obj
    elif df_name == 'test_1_pl':
        test_1_pl = df_obj
    elif df_name == 'test_2_pl':
        test_2_pl = df_obj
    elif df_name == 'test_3_pl':
        test_3_pl = df_obj
    elif df_name == 'test_4_pl':
        test_4_pl = df_obj
    elif df_name == 'val':
        val = df_obj

    print(f"Finished imputing missing values in {df_name}.")

print("ExtraTrees imputation complete for all specified DataFrames.")

In [ ]:
all_datasets = [
    (train_pl, 'train'),
    (val, 'val'),
    (test_1_pl, 'test_1'),
    (test_2_pl, 'test_2'),
    (test_3_pl, 'test_3'),
    (test_4_pl, 'test_4')
]

for df_obj, df_name in all_datasets:
    print(f"Processing {df_name}...")

    y = df_obj['label_3']

    cols_to_drop = ['user_id', 'course_id', 'label_3']

    X = df_obj.drop(columns=cols_to_drop)

    globals()[f'X_{df_name}'] = X
    globals()[f'y_{df_name}'] = y

    print(f"Finished processing {df_name}. X_{df_name} shape: {X.shape}, y_{df_name} shape: {y.shape}")

print("Feature and target separation complete for all datasets.")

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

print("Training the model...")
model.fit(X_train, y_train)
print("Model trained successfully.")

In [ ]:
print("Making predictions on validation and test sets using NumPy arrays...")

y_pred_val = model.predict(X_val)
y_pred_test_1 = model.predict(X_test_1)
y_pred_test_2 = model.predict(X_test_2)
y_pred_test_3 = model.predict(X_test_3)
y_pred_test_4 = model.predict(X_test_4)

print("Predictions made successfully on all datasets with NumPy arrays.")

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

y_pred_val_rounded = np.round(y_pred_val)
y_pred_test_1_rounded = np.round(y_pred_test_1)
y_pred_test_2_rounded = np.round(y_pred_test_2)
y_pred_test_3_rounded = np.round(y_pred_test_3)
y_pred_test_4_rounded = np.round(y_pred_test_4)

print('Accuracy on validation set:', accuracy_score(y_val, y_pred_val_rounded))
print('Accuracy on test set 1:', accuracy_score(y_test_1, y_pred_test_1_rounded))
print('Accuracy on test set 2:', accuracy_score(y_test_2, y_pred_test_2_rounded))
print('Accuracy on test set 3:', accuracy_score(y_test_3, y_pred_test_3_rounded))
print('Accuracy on test set 4:', accuracy_score(y_test_4, y_pred_test_4_rounded))

In [ ]:
X_train

In [ ]:
import os

output_path = '/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_extra_trees_im_3/'
os.makedirs(output_path, exist_ok=True)

train_pl.to_csv(f'{output_path}/train.csv', index=False)
val.to_csv(f'{output_path}/val.csv', index=False)
test_1_pl.to_csv(f'{output_path}/test_1.csv', index=False)
test_2_pl.to_csv(f'{output_path}/test_2.csv', index=False)
test_3_pl.to_csv(f'{output_path}/test_3.csv', index=False)
test_4_pl.to_csv(f'{output_path}/test_4.csv', index=False)